# Warehouse arrivals validation queries

In [ ]:
catalog = "workspace"
schema = "dev_marek_prihoda_warehouse_arrivals"
volume = "raw_data"

volume_base = f"/Volumes/{catalog}/{schema}/{volume}"
gps_root = f"{volume_base}/gps"
geofences_path = f"{volume_base}/geofences"

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Running validation queries in {catalog}.{schema}")
print(f"Checking managed volume data under {volume_base}")

def fq(table_name: str) -> str:
    return f"`{catalog}`.`{schema}`.`{table_name}`"

def volume_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def list_volume_path(path: str):
    if not volume_exists(path):
        return spark.createDataFrame([(path, '<missing>', 0, None)], ['path', 'name', 'size', 'modificationTime'])

    return spark.createDataFrame(
        [(f.path, f.name, f.size, getattr(f, 'modificationTime', None)) for f in dbutils.fs.ls(path)],
        ['path', 'name', 'size', 'modificationTime'],
    )


## Raw data volume checks

In [ ]:
display(list_volume_path(volume_base))
display(list_volume_path(gps_root))
display(list_volume_path(geofences_path))


In [ ]:
if volume_exists(gps_root):
    spark.read.option('recursiveFileLookup', 'true').json(gps_root).createOrReplaceTempView('gps_volume_raw')
else:
    spark.range(0).selectExpr(
        'CAST(NULL AS STRING) AS device_id',
        'CAST(NULL AS TIMESTAMP) AS timestamp',
        'CAST(NULL AS DOUBLE) AS longitude',
        'CAST(NULL AS DOUBLE) AS latitude'
    ).createOrReplaceTempView('gps_volume_raw')

if volume_exists(geofences_path):
    spark.read.json(geofences_path).createOrReplaceTempView('geofences_volume_raw')
else:
    spark.range(0).selectExpr(
        'CAST(NULL AS STRING) AS warehouse_name',
        'CAST(NULL AS STRING) AS boundary_wkt'
    ).createOrReplaceTempView('geofences_volume_raw')


## Raw volume row counts and time range

In [ ]:
display(spark.sql("""
SELECT 'gps_volume_raw' AS dataset_name, COUNT(*) AS row_count, MIN(timestamp) AS min_timestamp, MAX(timestamp) AS max_timestamp
FROM gps_volume_raw
UNION ALL
SELECT 'geofences_volume_raw' AS dataset_name, COUNT(*) AS row_count, CAST(NULL AS TIMESTAMP) AS min_timestamp, CAST(NULL AS TIMESTAMP) AS max_timestamp
FROM geofences_volume_raw
"""))


## Sample records directly from the volume

In [ ]:
display(spark.sql("""
SELECT warehouse_name, boundary_wkt
FROM geofences_volume_raw
ORDER BY warehouse_name
"""))

display(spark.sql("""
SELECT device_id, timestamp, longitude, latitude
FROM gps_volume_raw
ORDER BY timestamp DESC, device_id
LIMIT 20
"""))


## Row counts across pipeline tables

In [ ]:
display(spark.sql(f"""
SELECT 'gps_bronze' AS table_name, COUNT(*) AS row_count FROM {fq('gps_bronze')}
UNION ALL
SELECT 'geofences_bronze' AS table_name, COUNT(*) AS row_count FROM {fq('geofences_bronze')}
UNION ALL
SELECT 'raw_gps_silver' AS table_name, COUNT(*) AS row_count FROM {fq('raw_gps_silver')}
UNION ALL
SELECT 'warehouse_geofences_gold' AS table_name, COUNT(*) AS row_count FROM {fq('warehouse_geofences_gold')}
UNION ALL
SELECT 'warehouse_arrivals_gold' AS table_name, COUNT(*) AS row_count FROM {fq('warehouse_arrivals_gold')}
ORDER BY table_name
"""))


## Arrivals by warehouse

In [ ]:
display(spark.sql(f"""
SELECT warehouse_name, COUNT(*) AS arrival_count
FROM {fq('warehouse_arrivals_gold')}
GROUP BY warehouse_name
ORDER BY warehouse_name
"""))


## Most recent arrivals

In [ ]:
display(spark.sql(f"""
SELECT device_id, timestamp, warehouse_name
FROM {fq('warehouse_arrivals_gold')}
ORDER BY timestamp DESC
LIMIT 10
"""))


## Warehouse coverage and arrival window

In [ ]:
display(spark.sql(f"""
SELECT
  warehouse_name,
  COUNT(*) AS arrival_count,
  COUNT(DISTINCT device_id) AS distinct_devices,
  MIN(timestamp) AS first_arrival_ts,
  MAX(timestamp) AS last_arrival_ts
FROM {fq('warehouse_arrivals_gold')}
GROUP BY warehouse_name
ORDER BY warehouse_name
"""))


## GPS pings with no matching warehouse geofence

In [ ]:
display(spark.sql(f"""
SELECT COUNT(*) AS gps_points_without_match
FROM {fq('raw_gps_silver')} AS g
LEFT JOIN {fq('warehouse_geofences_gold')} AS w
  ON ST_Contains(w.boundary_geom, g.point_geom)
WHERE w.warehouse_name IS NULL
"""))


## Potential duplicate matches caused by overlapping geofences

In [ ]:
display(spark.sql(f"""
SELECT
  device_id,
  timestamp,
  COUNT(*) AS matched_warehouse_count,
  SORT_ARRAY(COLLECT_SET(warehouse_name)) AS matched_warehouses
FROM {fq('warehouse_arrivals_gold')}
GROUP BY device_id, timestamp
HAVING COUNT(*) > 1
ORDER BY matched_warehouse_count DESC, timestamp DESC
LIMIT 20
"""))
